# 19. 18번 log 0 경고 점검 — T_phys300의 NaN 처리가 결과를 바꿨나

경고 원인: 팀원 물리 관계 R1·R2의 `log(0)`, R4의 0 나눗셈 → NaN → **그 샷 T_phys300 점수 = 0 (가장 정상)**

1. 어떤 데이터·변수에 0이 있나
2. labeled 샷 중 영향 받은 샷 (불량 포함 여부)
3. NaN 처리만 바꿔 성능 비교: `skip`(18번 그대로) vs `max`(계산 불가 관계 = z 20)

**판정**: 2에서 영향 받은 불량이 0개이고 3에서 AP 차이가 거의 없으면 → 18번 결과 그대로 사용

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np, pandas as pd
from compare_frames import (load_all_shots, load_unlabeled_normal, official_rows, _past_base,
                            my_scores, frame1_eval, frame2_oof, frame2_eval)
from final_compare import combine
from model_phys import add_phys
from scope import apply_scope
from check_logzero import zero_table, nan_relation_table, affected_labeled, phys_scores
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
OUT = './outputs/final_compare'
os.makedirs(OUT, exist_ok=True)
PRODUCTS = ['cn7', 'rg3']
REF_CACHE = './base/ref_{}_features_nozero_2020-09-01_2020-11-30.csv'

SH, UNL, FEAT, REF = {}, {}, {}, {}
for p in PRODUCTS:
    SH[p] = load_all_shots(p)
    UNL[p] = load_unlabeled_normal(p)
    f = add_phys(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']))
    f['Reason'] = f['Reason'].fillna('정상')
    FEAT[p] = f
    REF[p] = add_phys(pd.read_csv(REF_CACHE.format(p), parse_dates=['TimeStamp']))

[cn7] 비라벨 28744샷 중 0 모드 18139 (63.1%) 제거
[rg3] 비라벨 29633샷 중 0 모드 13119 (44.3%) 제거


## 1. 0 값 개수 / 관계별 NaN 개수

In [2]:
Z, N = [], []
for p in PRODUCTS:
    ds = {'labeled 샷': SH[p], 'unlabeled(0모드 제거)': UNL[p], 'reference': REF[p]}
    Z.append(zero_table(ds).assign(제품=p))
    N.append(nan_relation_table(ds).assign(제품=p))
Z = pd.concat(Z, ignore_index=True); N = pd.concat(N, ignore_index=True)
display(Z[Z['0 이하'] > 0].round(4) if (Z['0 이하'] > 0).any() else '0 이하 값 없음')
display(N)

c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,데이터,변수,샷 수,0 이하,비율,NaN,제품
9,unlabeled(0모드 제거),Max_Screw_RPM,10605,1,0.0001,0,cn7
24,unlabeled(0모드 제거),Filling_Time,16514,1,0.0001,0,rg3
25,unlabeled(0모드 제거),Max_Injection_Speed,16514,1,0.0001,0,rg3
27,unlabeled(0모드 제거),Max_Screw_RPM,16514,3,0.0002,0,rg3
29,unlabeled(0모드 제거),Max_Injection_Pressure,16514,1,0.0001,0,rg3


,데이터,샷 수,관계 하나라도 NaN,R1 NaN,R2 NaN,R3 NaN,R4 NaN,R5 NaN,제품
0,labeled 샷,1979,0,0,0,0,0,0,cn7
1,unlabeled(0모드 제거),10605,1,0,1,0,0,0,cn7
2,reference,5179,0,0,0,0,0,0,cn7
3,labeled 샷,629,0,0,0,0,0,0,rg3
4,unlabeled(0모드 제거),16514,3,1,3,0,1,0,rg3
5,reference,4359,0,0,0,0,0,0,rg3


## 2. 영향 받은 labeled 샷 (점수가 0으로 처리된 샷)

In [3]:
AFF = {p: affected_labeled(SH[p]) for p in PRODUCTS}
for p in PRODUCTS:
    a = AFF[p]
    print(f'[{p}] 관계 NaN 샷 {len(a)} / 전체 {len(SH[p])}, 그중 불량 {int(a["PassOrFail"].sum())}')
    if len(a):
        display(a['Reason'].value_counts().rename('샷 수').to_frame())
        display(a[a['PassOrFail'] == 1])

ValueError: Cannot set a DataFrame with multiple columns to the single column NaN 관계

## 3. NaN 처리 방식별 성능 (skip = 18번 그대로)

In [ ]:
SC = {}
for p in PRODUCTS:
    sh, f, ref = SH[p], FEAT[p], REF[p]
    base = _past_base(sh, UNL[p])
    ts = f['TimeStamp'].values
    a_m, r_m = my_scores(f, ref)['M_maha_free']
    a_m = pd.Series(a_m, index=ts)
    S = {}
    for pol in ('skip', 'max'):
        s = pd.Series(phys_scores(sh, base, nan_policy=pol), index=sh['TimeStamp'].values)
        r = phys_scores(ref, base, nan_policy=pol)
        S[f'T_phys300({pol})'] = s
        c, _ = combine([(a_m.reindex(ts).fillna(0).values, r_m), (s.reindex(ts).fillna(0).values, r)])
        S[f'C_maha+phys300({pol})'] = pd.Series(c, index=ts)
        print(p, pol, '완료')
    SC[p] = S

F1T = pd.concat([frame1_eval(p, official_rows(p), SC[p]) for p in PRODUCTS], ignore_index=True)
O2 = pd.concat([frame2_oof(p, apply_scope(FEAT[p][FEAT[p]['split'] != '제외'], exclude_initial=True), SC[p])
                for p in PRODUCTS], ignore_index=True)
F2T = frame2_eval(O2)
F1T.to_csv(f'{OUT}/logzero_frame1.csv', index=False, encoding='utf-8-sig')
F2T.to_csv(f'{OUT}/logzero_frame2.csv', index=False, encoding='utf-8-sig')
print('① 공식 2차·행 단위'); display(F1T[['제품', '방법', 'AP', '유의', '상위5%', '상위10%', 'F1@5%']].round(3))
print('② 전체·샷 단위'); display(F2T[['product', 'method', 'AP', 'AP_lo', 'AP_hi', 'hit@10%', 'hit10_가스', 'AP_미성형']].round(3))

## 판정
- **skip 값이 18번 표의 T_phys300·C_maha+phys300과 같은지** 먼저 확인 (재현 확인)
- 2에서 영향 받은 불량 0개 + 3에서 skip ≈ max → **18번 결과 유지**, 보고서에 '0 값 샷 n개는 관계 계산 불가로 제외' 한 줄
- max가 뚜렷이 좋아지면 → 팀원 방식 수치를 max 기준으로 바꿔 18번 B(부트스트랩)만 다시 계산